In [1]:
import pandas as pd
import numpy as np
import re

# Ratings dataset
ratings_df = pd.read_csv(
    "../data/movies_dataset.csv"
)

# Movie metadata
imdb_df = pd.read_csv(
    "../data/imdb_movies.csv"
)

# Movie credits
credits_df = pd.read_csv(
    "../data/credits.csv"
)

# Movies used by our current ML model
model_movies = pd.read_csv(
    "../data/model_movies.csv"
)

print("Ratings:", ratings_df.shape)
print("IMDb metadata:", imdb_df.shape)
print("Credits:", credits_df.shape)
print("Model movies:", model_movies.shape)

Ratings: (25000095, 5)
IMDb metadata: (3853, 28)
Credits: (4803, 4)
Model movies: (3794, 1)


In [2]:
print("RATINGS")
print(ratings_df.columns.tolist())

print("\nIMDB METADATA")
print(imdb_df.columns.tolist())

print("\nCREDITS")
print(credits_df.columns.tolist())

print("\nMODEL MOVIES")
print(model_movies.columns.tolist())

RATINGS
['Unnamed: 0', 'User_Id', 'Movie_Name', 'Rating', 'Genre']

IMDB METADATA
['color', 'director_name', 'num_critic_for_reviews', 'duration', 'director_facebook_likes', 'actor_3_facebook_likes', 'actor_2_name', 'actor_1_facebook_likes', 'gross', 'genres', 'actor_1_name', 'movie_title', 'num_voted_users', 'cast_total_facebook_likes', 'actor_3_name', 'facenumber_in_poster', 'plot_keywords', 'movie_imdb_link', 'num_user_for_reviews', 'language', 'country', 'content_rating', 'budget', 'title_year', 'actor_2_facebook_likes', 'imdb_score', 'aspect_ratio', 'movie_facebook_likes']

CREDITS
['movie_id', 'title', 'cast', 'crew']

MODEL MOVIES
['Movie_Name']


In [3]:
rating_movies = (
    ratings_df[["Movie_Name"]]
    .drop_duplicates()
    .copy()
)

print("Unique rating movies:", len(rating_movies))

Unique rating movies: 58958


In [4]:
def normalize_title(title):
    title = str(title).lower().strip()

    # Convert "movie, the" → "the movie"
    if re.search(r",\s*the$", title):
        title = re.sub(r",\s*the$", "", title)
        title = "the " + title

    # Remove punctuation
    title = re.sub(r"[^a-z0-9\s]", " ", title)

    # Normalize whitespace
    title = re.sub(r"\s+", " ", title).strip()

    return title

In [5]:
test_titles = [
    "Shawshank Redemption, The",
    "The Shawshank Redemption",
    "Matrix, The",
    "The Matrix"
]

for title in test_titles:
    print(title, "→", normalize_title(title))

Shawshank Redemption, The → the shawshank redemption
The Shawshank Redemption → the shawshank redemption
Matrix, The → the matrix
The Matrix → the matrix


In [6]:
rating_movies["year"] = (
    rating_movies["Movie_Name"]
    .str.extract(r"\((\d{4})\)$")
)

rating_movies["clean_title"] = (
    rating_movies["Movie_Name"]
    .str.replace(
        r"\s*\(\d{4}\)$",
        "",
        regex=True
    )
    .str.strip()
)

rating_movies["year"] = pd.to_numeric(
    rating_movies["year"],
    errors="coerce"
).astype("Int64")

rating_movies["normalized_title"] = (
    rating_movies["clean_title"]
    .apply(normalize_title)
)

rating_movies.head()

,Movie_Name,year,clean_title,normalized_title
0,Pulp Fiction (1994),1994,Pulp Fiction,pulp fiction
1,Three Colors: Red (Trois couleurs: Rouge) (1994),1994,Three Colors: Red (Trois couleurs: Rouge),three colors red trois couleurs rouge
2,Three Colors: Blue (Trois couleurs: Bleu) (1993),1993,Three Colors: Blue (Trois couleurs: Bleu),three colors blue trois couleurs bleu
3,Underground (1995),1995,Underground,underground
4,Singin' in the Rain (1952),1952,Singin' in the Rain,singin in the rain


In [7]:
model_movies["year"] = (
    model_movies["Movie_Name"]
    .str.extract(r"\((\d{4})\)$")
)

model_movies["clean_title"] = (
    model_movies["Movie_Name"]
    .str.replace(
        r"\s*\(\d{4}\)$",
        "",
        regex=True
    )
    .str.strip()
)

model_movies["year"] = pd.to_numeric(
    model_movies["year"],
    errors="coerce"
).astype("Int64")

model_movies["normalized_title"] = (
    model_movies["clean_title"]
    .apply(normalize_title)
)

model_movies.head()

,Movie_Name,year,clean_title,normalized_title
0,Toy Story (1995),1995,Toy Story,toy story
1,Mr. Holland's Opus (1995),1995,Mr. Holland's Opus,mr holland s opus
2,Braveheart (1995),1995,Braveheart,braveheart
3,Apollo 13 (1995),1995,Apollo 13,apollo 13
4,Rob Roy (1995),1995,Rob Roy,rob roy


In [8]:
print("Ratings:", ratings_df.shape)
print("IMDb metadata:", imdb_df.shape)
print("Credits:", credits_df.shape)
print("Model movies:", model_movies.shape)

Ratings: (25000095, 5)
IMDb metadata: (3853, 28)
Credits: (4803, 4)
Model movies: (3794, 4)


In [9]:
print(
    imdb_df[
        ["movie_title", "title_year"]
    ].head(20)
)

                                     movie_title  title_year
0                                        Avatar       2009.0
1      Pirates of the Caribbean: At World's End       2007.0
2                                       Spectre       2015.0
3                         The Dark Knight Rises       2012.0
4                                   John Carter       2012.0
5                                  Spider-Man 3       2007.0
6                                       Tangled       2010.0
7                       Avengers: Age of Ultron       2015.0
8        Harry Potter and the Half-Blood Prince       2009.0
9            Batman v Superman: Dawn of Justice       2016.0
10                             Superman Returns       2006.0
11                            Quantum of Solace       2008.0
12   Pirates of the Caribbean: Dead Man's Chest       2006.0
13                              The Lone Ranger       2013.0
14                                 Man of Steel       2013.0
15     The Chronicles of

In [10]:
print(imdb_df["movie_title"].isna().sum())
print(imdb_df["title_year"].isna().sum())

0
0


In [11]:
print(
    imdb_df[
        ["movie_title", "title_year"]
    ].tail(10)
)

             movie_title  title_year
3843   Breaking Upwards       2009.0
3844     Pink Flamingos       1972.0
3845              Clean       2004.0
3846         The Circle       2000.0
3847           The Cure       1997.0
3848             Primer       2004.0
3849             Cavite       2005.0
3850        El Mariachi       1992.0
3851          Newlyweds       2011.0
3852  My Date with Drew       2004.0


In [12]:
# Prepare IMDb titles for matching

imdb_df["year"] = (
    pd.to_numeric(imdb_df["title_year"], errors="coerce")
    .astype("Int64")
)

imdb_df["clean_title"] = (
    imdb_df["movie_title"]
    .astype(str)
    .str.strip()
)

imdb_df["normalized_title"] = (
    imdb_df["clean_title"]
    .apply(normalize_title)
)

imdb_df[
    ["movie_title", "year", "normalized_title"]
].head(10)

,movie_title,year,normalized_title
0,Avatar,2009,avatar
1,Pirates of the Caribbean: At World's End,2007,pirates of the caribbean at world s end
2,Spectre,2015,spectre
3,The Dark Knight Rises,2012,the dark knight rises
4,John Carter,2012,john carter
5,Spider-Man 3,2007,spider man 3
6,Tangled,2010,tangled
7,Avengers: Age of Ultron,2015,avengers age of ultron
8,Harry Potter and the Half-Blood Prince,2009,harry potter and the half blood prince
9,Batman v Superman: Dawn of Justice,2016,batman v superman dawn of justice


In [13]:
rating_imdb_matches = rating_movies.merge(
    imdb_df[
        ["normalized_title", "year", "movie_title"]
    ],
    on=["normalized_title", "year"],
    how="inner"
)

print("Ratings movies:", len(rating_movies))
print("IMDb matches:", len(rating_imdb_matches))

print(
    "Match percentage:",
    round(
        len(rating_imdb_matches) / len(rating_movies) * 100,
        2
    ),
    "%"
)

Ratings movies: 58958
IMDb matches: 3448
Match percentage: 5.85 %


In [14]:
rating_imdb_matches[
    ["Movie_Name", "movie_title", "year"]
].head(20)

,Movie_Name,movie_title,year
0,Pulp Fiction (1994),Pulp Fiction,1994
1,"Bridge on the River Kwai, The (1957)",The Bridge on the River Kwai,1957
2,Gattaca (1997),Gattaca,1997
3,Back to the Future Part II (1989),Back to the Future Part II,1989
4,Back to the Future Part III (1990),Back to the Future Part III,1990
5,Tango (1998),Tango,1998
6,"Good Morning, Vietnam (1987)","Good Morning, Vietnam",1987
7,Requiem for a Dream (2000),Requiem for a Dream,2000
8,Moulin Rouge (2001),Moulin Rouge!,2001
9,"Lord of the Rings: The Two Towers, The (2002)",The Lord of the Rings: The Two Towers,2002


In [15]:
imdb_columns = [
    "movie_title",
    "title_year",
    "genres",
    "director_name",
    "actor_1_name",
    "actor_2_name",
    "actor_3_name",
    "plot_keywords",
    "imdb_score",
    "num_voted_users",
    "movie_imdb_link"
]

imdb_clean = imdb_df[imdb_columns].copy()

print("Original IMDb columns:", len(imdb_df.columns))
print("Selected IMDb columns:", len(imdb_clean.columns))

imdb_clean.head()

Original IMDb columns: 31
Selected IMDb columns: 11


,movie_title,title_year,genres,director_name,actor_1_name,actor_2_name,actor_3_name,plot_keywords,imdb_score,num_voted_users,movie_imdb_link
0,Avatar,2009.0,Action|Adventure|Fantasy|Sci-Fi,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,avatar|future|marine|native|paraplegic,7.9,886204,http://www.imdb.com/title/tt0499549/?ref_=fn_t...
1,Pirates of the Caribbean: At World's End,2007.0,Action|Adventure|Fantasy,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,goddess|marriage ceremony|marriage proposal|pi...,7.1,471220,http://www.imdb.com/title/tt0449088/?ref_=fn_t...
2,Spectre,2015.0,Action|Adventure|Thriller,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,bomb|espionage|sequel|spy|terrorist,6.8,275868,http://www.imdb.com/title/tt2379713/?ref_=fn_t...
3,The Dark Knight Rises,2012.0,Action|Thriller,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,deception|imprisonment|lawlessness|police offi...,8.5,1144337,http://www.imdb.com/title/tt1345836/?ref_=fn_t...
4,John Carter,2012.0,Action|Adventure|Sci-Fi,Andrew Stanton,Daryl Sabara,Samantha Morton,Polly Walker,alien|american civil war|male nipple|mars|prin...,6.6,212204,http://www.imdb.com/title/tt0401729/?ref_=fn_t...


In [16]:
imdb_clean = imdb_clean.rename(columns={
    "movie_title": "title",
    "title_year": "year",
    "director_name": "director",
    "actor_1_name": "actor_1",
    "actor_2_name": "actor_2",
    "actor_3_name": "actor_3",
    "plot_keywords": "keywords",
    "imdb_score": "imdb_rating",
    "num_voted_users": "imdb_votes",
    "movie_imdb_link": "imdb_link"
})

imdb_clean.head()

,title,year,genres,director,actor_1,actor_2,actor_3,keywords,imdb_rating,imdb_votes,imdb_link
0,Avatar,2009.0,Action|Adventure|Fantasy|Sci-Fi,James Cameron,CCH Pounder,Joel David Moore,Wes Studi,avatar|future|marine|native|paraplegic,7.9,886204,http://www.imdb.com/title/tt0499549/?ref_=fn_t...
1,Pirates of the Caribbean: At World's End,2007.0,Action|Adventure|Fantasy,Gore Verbinski,Johnny Depp,Orlando Bloom,Jack Davenport,goddess|marriage ceremony|marriage proposal|pi...,7.1,471220,http://www.imdb.com/title/tt0449088/?ref_=fn_t...
2,Spectre,2015.0,Action|Adventure|Thriller,Sam Mendes,Christoph Waltz,Rory Kinnear,Stephanie Sigman,bomb|espionage|sequel|spy|terrorist,6.8,275868,http://www.imdb.com/title/tt2379713/?ref_=fn_t...
3,The Dark Knight Rises,2012.0,Action|Thriller,Christopher Nolan,Tom Hardy,Christian Bale,Joseph Gordon-Levitt,deception|imprisonment|lawlessness|police offi...,8.5,1144337,http://www.imdb.com/title/tt1345836/?ref_=fn_t...
4,John Carter,2012.0,Action|Adventure|Sci-Fi,Andrew Stanton,Daryl Sabara,Samantha Morton,Polly Walker,alien|american civil war|male nipple|mars|prin...,6.6,212204,http://www.imdb.com/title/tt0401729/?ref_=fn_t...


In [17]:
credits_clean = credits_df[
    ["movie_id", "title", "cast", "crew"]
].copy()

credits_clean = credits_clean.rename(columns={
    "movie_id": "tmdb_id"
})

credits_clean["tmdb_id"] = (
    credits_clean["tmdb_id"]
    .astype("int64")
)

credits_clean.head()

,tmdb_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [18]:
ratings_clean = ratings_df[
    ["User_Id", "Movie_Name", "Rating", "Genre"]
].copy()

ratings_clean.head()

,User_Id,Movie_Name,Rating,Genre
0,1,Pulp Fiction (1994),5.0,Comedy|Crime|Drama|Thriller
1,1,Three Colors: Red (Trois couleurs: Rouge) (1994),3.5,Drama
2,1,Three Colors: Blue (Trois couleurs: Bleu) (1993),5.0,Drama
3,1,Underground (1995),5.0,Comedy|Drama|War
4,1,Singin' in the Rain (1952),3.5,Comedy|Musical|Romance


In [19]:
model_movies_clean = model_movies[
    ["Movie_Name"]
].drop_duplicates().copy()

print("Model movies:", len(model_movies_clean))

Model movies: 3794


In [20]:
canonical_movies = model_movies_clean.copy()

canonical_movies["year"] = (
    canonical_movies["Movie_Name"]
    .str.extract(r"\((\d{4})\)$")
)

canonical_movies["year"] = pd.to_numeric(
    canonical_movies["year"],
    errors="coerce"
).astype("Int64")

canonical_movies["title"] = (
    canonical_movies["Movie_Name"]
    .str.replace(
        r"\s*\(\d{4}\)$",
        "",
        regex=True
    )
    .str.strip()
)

canonical_movies["normalized_title"] = (
    canonical_movies["title"]
    .apply(normalize_title)
)

canonical_movies.head()

,Movie_Name,year,title,normalized_title
0,Toy Story (1995),1995,Toy Story,toy story
1,Mr. Holland's Opus (1995),1995,Mr. Holland's Opus,mr holland s opus
2,Braveheart (1995),1995,Braveheart,braveheart
3,Apollo 13 (1995),1995,Apollo 13,apollo 13
4,Rob Roy (1995),1995,Rob Roy,rob roy


In [21]:
canonical_movies = canonical_movies.reset_index(drop=True)

canonical_movies["canonical_movie_id"] = (
    canonical_movies.index + 1
)

In [22]:
canonical_movies = canonical_movies[
    [
        "canonical_movie_id",
        "Movie_Name",
        "title",
        "year",
        "normalized_title"
    ]
]

canonical_movies.head(10)

,canonical_movie_id,Movie_Name,title,year,normalized_title
0,1,Toy Story (1995),Toy Story,1995,toy story
1,2,Mr. Holland's Opus (1995),Mr. Holland's Opus,1995,mr holland s opus
2,3,Braveheart (1995),Braveheart,1995,braveheart
3,4,Apollo 13 (1995),Apollo 13,1995,apollo 13
4,5,Rob Roy (1995),Rob Roy,1995,rob roy
5,6,French Kiss (1995),French Kiss,1995,french kiss
6,7,Star Wars: Episode IV - A New Hope (1977),Star Wars: Episode IV - A New Hope,1977,star wars episode iv a new hope
7,8,Little Women (1994),Little Women,1994,little women
8,9,Legends of the Fall (1994),Legends of the Fall,1994,legends of the fall
9,10,"Shawshank Redemption, The (1994)","Shawshank Redemption, The",1994,the shawshank redemption


In [23]:
print(canonical_movies.shape)
print(canonical_movies.head())

(3794, 5)
   canonical_movie_id                 Movie_Name               title  year  \
0                   1           Toy Story (1995)           Toy Story  1995   
1                   2  Mr. Holland's Opus (1995)  Mr. Holland's Opus  1995   
2                   3          Braveheart (1995)          Braveheart  1995   
3                   4           Apollo 13 (1995)           Apollo 13  1995   
4                   5             Rob Roy (1995)             Rob Roy  1995   

    normalized_title  
0          toy story  
1  mr holland s opus  
2         braveheart  
3          apollo 13  
4            rob roy  


In [24]:
# Prepare IMDb data for matching

imdb_match = imdb_clean.copy()

imdb_match["normalized_title"] = (
    imdb_match["title"]
    .fillna("")
    .apply(normalize_title)
)

imdb_match["year"] = pd.to_numeric(
    imdb_match["year"],
    errors="coerce"
).astype("Int64")

print("IMDb movies:", len(imdb_match))

IMDb movies: 3853


In [25]:
canonical_imdb = canonical_movies.merge(
    imdb_match,
    on=["normalized_title", "year"],
    how="left",
    suffixes=("", "_imdb")
)

print("Canonical movies:", len(canonical_movies))
print(
    "IMDb matched:",
    canonical_imdb["imdb_rating"].notna().sum()
)

Canonical movies: 3794
IMDb matched: 1969


In [26]:
imdb_coverage = (
    canonical_imdb["imdb_rating"].notna().sum()
    / len(canonical_imdb)
    * 100
)

print(
    f"IMDb metadata coverage: {imdb_coverage:.2f}%"
)

IMDb metadata coverage: 51.24%


In [27]:
credits_clean["normalized_title"] = (
    credits_clean["title"]
    .fillna("")
    .apply(normalize_title)
)

In [28]:
canonical_credits = canonical_imdb.merge(
    credits_clean[
        ["tmdb_id", "normalized_title", "cast", "crew"]
    ],
    on="normalized_title",
    how="left"
)

In [29]:
canonical_credits = canonical_imdb.merge(
    credits_clean[
        ["tmdb_id", "normalized_title", "cast", "crew"]
    ],
    on="normalized_title",
    how="left"
)

In [30]:
print("Before credits merge:", len(canonical_imdb))
print("After credits merge:", len(canonical_credits))

Before credits merge: 3843
After credits merge: 3846


In [31]:
imdb_duplicates = (
    canonical_imdb[
        canonical_imdb["normalized_title"].duplicated(keep=False)
    ]
    .sort_values("normalized_title")
)

print("Duplicate normalized titles:", len(imdb_duplicates))

imdb_duplicates[
    ["canonical_movie_id", "Movie_Name", "title", "year"]
].head(30)

Duplicate normalized titles: 221


,canonical_movie_id,Movie_Name,title,year
2210,2184,Across the Universe (2007),Across the Universe,2007
2209,2184,Across the Universe (2007),Across the Universe,2007
966,946,Alice in Wonderland (1951),Alice in Wonderland,1951
581,571,Alice in Wonderland (2010),Alice in Wonderland,2010
582,571,Alice in Wonderland (2010),Alice in Wonderland,2010
1047,1026,Around the World in 80 Days (1956),Around the World in 80 Days,1956
3787,3740,Around the World in 80 Days (2004),Around the World in 80 Days,2004
3786,3740,Around the World in 80 Days (2004),Around the World in 80 Days,2004
2164,2139,Assault on Precinct 13 (2005),Assault on Precinct 13,2005
2140,2115,Assault on Precinct 13 (1976),Assault on Precinct 13,1976


In [32]:
duplicate_ids = (
    canonical_imdb["canonical_movie_id"]
    .value_counts()
)

duplicate_ids = duplicate_ids[duplicate_ids > 1]

print("Canonical movies with multiple IMDb matches:")
print(duplicate_ids)

Canonical movies with multiple IMDb matches:
canonical_movie_id
441     3
2860    3
592     2
779     2
467     2
284     2
391     2
638     2
1285    2
649     2
289     2
59      2
678     2
571     2
456     2
438     2
1537    2
792     2
953     2
645     2
635     2
478     2
1479    2
591     2
1987    2
2184    2
2395    2
2398    2
2442    2
2511    2
2514    2
2526    2
2670    2
428     2
2878    2
3113    2
3197    2
3283    2
3294    2
3453    2
3499    2
3511    2
3564    2
3592    2
3621    2
3740    2
3749    2
Name: count, dtype: int64


In [33]:
on=["normalized_title", "year"]

In [34]:
# Create a one-row-per-title/year IMDb lookup table

imdb_lookup = (
    imdb_match
    .drop_duplicates(
        subset=["normalized_title", "year"],
        keep="first"
    )
    .copy()
)

print("Original IMDb rows:", len(imdb_match))
print("Unique IMDb title/year rows:", len(imdb_lookup))

Original IMDb rows: 3853
Unique IMDb title/year rows: 3786


In [35]:
print(
    "Duplicate title/year keys:",
    imdb_lookup.duplicated(
        subset=["normalized_title", "year"]
    ).sum()
)

Duplicate title/year keys: 0


In [36]:
canonical_imdb = canonical_movies.merge(
    imdb_lookup,
    on=["normalized_title", "year"],
    how="left",
    suffixes=("", "_imdb")
)

print("Canonical movies:", len(canonical_movies))
print("After IMDb merge:", len(canonical_imdb))

Canonical movies: 3794
After IMDb merge: 3794


In [37]:
imdb_coverage = (
    canonical_imdb["imdb_rating"].notna().sum()
    / len(canonical_imdb)
    * 100
)

print(f"IMDb metadata coverage: {imdb_coverage:.2f}%")

IMDb metadata coverage: 50.61%


In [38]:
credits_duplicates = (
    credits_clean[
        credits_clean["normalized_title"].duplicated(keep=False)
    ]
    .sort_values("normalized_title")
)

print(
    "Credits rows with duplicate titles:",
    len(credits_duplicates)
)

credits_duplicates[
    ["tmdb_id", "title", "normalized_title"]
].head(30)

Credits rows with duplicate titles: 10


,tmdb_id,title,normalized_title
838,8077,Alien³,alien
3158,348,Alien,alien
1359,268,Batman,batman
4267,2661,Batman,batman
3647,39269,Out of the Blue,out of the blue
3693,10844,Out of the Blue,out of the blue
3524,10664,[REC]²,rec
4066,8329,[REC],rec
972,72710,The Host,the host
2877,1255,The Host,the host


In [39]:
print(
    "Unique credits titles:",
    credits_clean["normalized_title"].nunique()
)

print(
    "Total credits rows:",
    len(credits_clean)
)

Unique credits titles: 4798
Total credits rows: 4803


In [40]:
# Keep only credits titles that occur exactly once

credits_title_counts = (
    credits_clean["normalized_title"]
    .value_counts()
)

unique_credit_titles = credits_title_counts[
    credits_title_counts == 1
].index

credits_lookup = credits_clean[
    credits_clean["normalized_title"].isin(unique_credit_titles)
].copy()

print("Original credits rows:", len(credits_clean))
print("Safe credits rows:", len(credits_lookup))
print(
    "Duplicate titles excluded:",
    len(credits_clean) - len(credits_lookup)
)

Original credits rows: 4803
Safe credits rows: 4793
Duplicate titles excluded: 10


In [41]:
canonical_master = canonical_imdb.merge(
    credits_lookup[
        ["tmdb_id", "normalized_title", "cast", "crew"]
    ],
    on="normalized_title",
    how="left"
)

In [42]:
print("Before credits:", len(canonical_imdb))
print("After credits:", len(canonical_master))

Before credits: 3794
After credits: 3794


In [43]:
credits_coverage = (
    canonical_master["tmdb_id"].notna().sum()
    / len(canonical_master)
    * 100
)

print(
    f"Credits coverage: {credits_coverage:.2f}%"
)

Credits coverage: 56.85%


In [44]:
canonical_master[
    [
        "canonical_movie_id",
        "Movie_Name",
        "year",
        "imdb_rating",
        "director",
        "actor_1",
        "actor_2",
        "actor_3",
        "tmdb_id"
    ]
].head(20)

,canonical_movie_id,Movie_Name,year,imdb_rating,director,actor_1,actor_2,actor_3,tmdb_id
0,1,Toy Story (1995),1995,8.3,John Lasseter,Tom Hanks,John Ratzenberger,Jim Varney,862.0
1,2,Mr. Holland's Opus (1995),1995,7.3,Stephen Herek,Alicia Witt,Olympia Dukakis,Jean Louisa Kelly,2054.0
2,3,Braveheart (1995),1995,8.4,Mel Gibson,Mhairi Calvey,Patrick McGoohan,James Robinson,197.0
3,4,Apollo 13 (1995),1995,7.6,Ron Howard,Tom Hanks,Miko Hughes,Kathleen Quinlan,568.0
4,5,Rob Roy (1995),1995,6.9,Michael Caton-Jones,Liam Neeson,Jason Flemyng,Eric Stoltz,11780.0
5,6,French Kiss (1995),1995,NaN,NaN,NaN,NaN,NaN,NaN
6,7,Star Wars: Episode IV - A New Hope (1977),1977,8.7,George Lucas,Harrison Ford,Peter Cushing,Kenny Baker,NaN
7,8,Little Women (1994),1994,7.3,Gillian Armstrong,Christian Bale,Kirsten Dunst,Eric Stoltz,9587.0
8,9,Legends of the Fall (1994),1994,7.5,Edward Zwick,Anthony Hopkins,Brad Pitt,Julia Ormond,4476.0
9,10,"Shawshank Redemption, The (1994)",1994,9.3,Frank Darabont,Morgan Freeman,Jeffrey DeMunn,Bob Gunton,278.0


In [45]:
print("Total movies:", len(canonical_master))

print(
    "IMDb available:",
    canonical_master["imdb_rating"].notna().sum()
)

print(
    "TMDB available:",
    canonical_master["tmdb_id"].notna().sum()
)

print(
    "Both IMDb + TMDB:",
    (
        canonical_master["imdb_rating"].notna()
        &
        canonical_master["tmdb_id"].notna()
    ).sum()
)

print(
    "No external metadata:",
    (
        canonical_master["imdb_rating"].isna()
        &
        canonical_master["tmdb_id"].isna()
    ).sum()
)

Total movies: 3794
IMDb available: 1920
TMDB available: 2157
Both IMDb + TMDB: 1910
No external metadata: 1627


In [46]:
def metadata_status(row):
    has_imdb = pd.notna(row["imdb_rating"])
    has_tmdb = pd.notna(row["tmdb_id"])

    if has_imdb and has_tmdb:
        return "IMDb + TMDB"

    elif has_imdb:
        return "IMDb only"

    elif has_tmdb:
        return "TMDB only"

    else:
        return "No metadata"

In [47]:
canonical_master["metadata_status"] = (
    canonical_master.apply(
        metadata_status,
        axis=1
    )
)

print(
    canonical_master["metadata_status"]
    .value_counts()
)

metadata_status
IMDb + TMDB    1910
No metadata    1627
TMDB only       247
IMDb only        10
Name: count, dtype: int64


In [48]:
# Final movie master table

movie_master = canonical_master.copy()

print("Movie master shape:", movie_master.shape)
print("Unique movie IDs:", movie_master["canonical_movie_id"].nunique())

Movie master shape: (3794, 19)
Unique movie IDs: 3794


In [49]:
movie_master.to_csv(
    "../data/movie_master.csv",
    index=False
)

print("Movie master saved successfully!")

Movie master saved successfully!


In [50]:
print("model exists:", "model" in globals())
print("movie_encoder exists:", "movie_encoder" in globals())

print("\nPossible model variables:")
print([
    name for name in globals()
    if "model" in name.lower()
    or "neighbor" in name.lower()
    or "knn" in name.lower()
])

print("\nPossible encoder variables:")
print([
    name for name in globals()
    if "encoder" in name.lower()
])

model exists: False
movie_encoder exists: False

Possible model variables:
['model_movies', 'model_movies_clean']

Possible encoder variables:
[]


In [51]:
print("model exists:", "model" in globals())
print("movie_encoder exists:", "movie_encoder" in globals())

print("\nPossible model variables:")
print([
    name for name in globals()
    if "model" in name.lower()
    or "neighbor" in name.lower()
    or "knn" in name.lower()
])

print("\nPossible encoder variables:")
print([
    name for name in globals()
    if "encoder" in name.lower()
])

model exists: False
movie_encoder exists: False

Possible model variables:
['model_movies', 'model_movies_clean']

Possible encoder variables:
[]


In [52]:
print("filtered_df exists:", "filtered_df" in globals())
print("sparse_matrix exists:", "sparse_matrix" in globals())
print("user_encoder exists:", "user_encoder" in globals())
print("movie_encoder exists:", "movie_encoder" in globals())

filtered_df exists: False
sparse_matrix exists: False
user_encoder exists: False
movie_encoder exists: False


In [53]:
import pandas as pd
import numpy as np

ratings_ml = pd.read_csv(
    "../data/movies_dataset.csv",
    usecols=["User_Id", "Movie_Name", "Rating"]
)

print("Ratings loaded:", ratings_ml.shape)
print("Unique users:", ratings_ml["User_Id"].nunique())
print("Unique movies:", ratings_ml["Movie_Name"].nunique())

Ratings loaded: (25000095, 3)
Unique users: 162541
Unique movies: 58958


In [54]:
model_movie_names = set(
    model_movies_clean["Movie_Name"]
)

ratings_ml = ratings_ml[
    ratings_ml["Movie_Name"].isin(model_movie_names)
].copy()

print("Ratings after model-movie filtering:", ratings_ml.shape)
print(
    "Movies remaining:",
    ratings_ml["Movie_Name"].nunique()
)

Ratings after model-movie filtering: (22142587, 3)
Movies remaining: 3794


In [55]:
from sklearn.preprocessing import LabelEncoder

user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

ratings_ml["user_index"] = user_encoder.fit_transform(
    ratings_ml["User_Id"]
)

ratings_ml["movie_index"] = movie_encoder.fit_transform(
    ratings_ml["Movie_Name"]
)

print("Users:", len(user_encoder.classes_))
print("Movies:", len(movie_encoder.classes_))

Users: 162539
Movies: 3794


In [56]:
from scipy.sparse import csr_matrix

sparse_matrix = csr_matrix(
    (
        ratings_ml["Rating"].astype(np.float32),
        (
            ratings_ml["user_index"],
            ratings_ml["movie_index"]
        )
    ),
    shape=(
        len(user_encoder.classes_),
        len(movie_encoder.classes_)
    )
)

print("Sparse matrix shape:", sparse_matrix.shape)
print("Non-zero ratings:", sparse_matrix.nnz)

Sparse matrix shape: (162539, 3794)
Non-zero ratings: 22142110


In [57]:
from sklearn.neighbors import NearestNeighbors

model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11
)

model.fit(sparse_matrix.T)

print("Model trained successfully!")

Model trained successfully!


In [58]:
# Connect model movie indices to our canonical movie IDs

model_movie_map = pd.DataFrame({
    "movie_index": range(len(movie_encoder.classes_)),
    "Movie_Name": movie_encoder.classes_
})

model_movie_map = model_movie_map.merge(
    movie_master[
        ["canonical_movie_id", "Movie_Name"]
    ],
    on="Movie_Name",
    how="left"
)

print("Model movies:", len(model_movie_map))
print(
    "Canonical IDs found:",
    model_movie_map["canonical_movie_id"].notna().sum()
)

model_movie_map.head(10)

Model movies: 3794
Canonical IDs found: 3794


,movie_index,Movie_Name,canonical_movie_id
0,0,"'burbs, The (1989)",1880
1,1,(500) Days of Summer (2009),1485
2,2,*batteries not included (1987),3739
3,3,...And Justice for All (1979),1889
4,4,10 Cloverfield Lane (2016),2850
5,5,10 Things I Hate About You (1999),1090
6,6,"10,000 BC (2008)",2742
7,7,101 Dalmatians (1996),1071
8,8,101 Dalmatians (One Hundred and One Dalmatians...,1087
9,9,102 Dalmatians (2000),260


In [59]:
def recommend_with_metadata(movie_name, n_recommendations=10):

    # Check if movie exists in the model
    if movie_name not in movie_encoder.classes_:
        print("Movie not found in the recommendation model.")
        return None

    # Get movie index
    movie_index = movie_encoder.transform([movie_name])[0]

    # Find similar movies
    distances, indices = model.kneighbors(
        sparse_matrix.T[movie_index],
        n_neighbors=n_recommendations + 1
    )

    recommendations = []

    for distance, index in zip(
        distances[0][1:],
        indices[0][1:]
    ):
        recommended_movie = movie_encoder.inverse_transform(
            [index]
        )[0]

        similarity = 1 - distance

        recommendations.append({
            "Movie_Name": recommended_movie,
            "Similarity": similarity
        })

    # Convert recommendations to DataFrame
    recommendations_df = pd.DataFrame(recommendations)

    # Attach canonical movie ID
    recommendations_df = recommendations_df.merge(
        movie_master,
        on="Movie_Name",
        how="left"
    )

    return recommendations_df

In [60]:
recommendations = recommend_with_metadata(
    "Toy Story (1995)",
    n_recommendations=10
)

recommendations[
    [
        "Movie_Name",
        "Similarity",
        "year",
        "imdb_rating",
        "director",
        "metadata_status"
    ]
]

,Movie_Name,Similarity,year,imdb_rating,director,metadata_status
0,Star Wars: Episode IV - A New Hope (1977),0.567278,1977,8.7,George Lucas,IMDb only
1,Toy Story 2 (1999),0.563549,1999,7.9,John Lasseter,IMDb + TMDB
2,Back to the Future (1985),0.548156,1985,8.5,Robert Zemeckis,IMDb + TMDB
3,Forrest Gump (1994),0.543782,1994,8.8,Robert Zemeckis,IMDb + TMDB
4,Jurassic Park (1993),0.539750,1993,8.1,Steven Spielberg,IMDb + TMDB
5,Star Wars: Episode VI - Return of the Jedi (1983),0.538781,1983,8.4,Richard Marquand,IMDb only
6,Independence Day (a.k.a. ID4) (1996),0.537721,1996,NaN,NaN,No metadata
7,"Lion King, The (1994)",0.530698,1994,8.5,Roger Allers,IMDb + TMDB
8,Aladdin (1992),0.528697,1992,8.0,Ron Clements,IMDb + TMDB
9,Star Wars: Episode V - The Empire Strikes Back...,0.513224,1980,8.8,Irvin Kershner,IMDb only


In [61]:
recommend_with_metadata(
    "Avatar (2099)",
    n_recommendations=5
)

Movie not found in the recommendation model.


In [62]:
def clean_recommendations(recommendations_df):

    columns = [
        "canonical_movie_id",
        "Movie_Name",
        "Similarity",
        "year",
        "genres",
        "imdb_rating",
        "director",
        "actor_1",
        "actor_2",
        "actor_3",
        "tmdb_id",
        "metadata_status"
    ]

    available_columns = [
        col for col in columns
        if col in recommendations_df.columns
    ]

    result = recommendations_df[available_columns].copy()

    result["Similarity"] = result["Similarity"].round(3)

    return result

In [63]:
clean_result = clean_recommendations(recommendations)

clean_result

,canonical_movie_id,Movie_Name,Similarity,year,genres,imdb_rating,director,actor_1,actor_2,actor_3,tmdb_id,metadata_status
0,7,Star Wars: Episode IV - A New Hope (1977),0.567,1977,Action|Adventure|Fantasy|Sci-Fi,8.7,George Lucas,Harrison Ford,Peter Cushing,Kenny Baker,NaN,IMDb only
1,105,Toy Story 2 (1999),0.564,1999,Adventure|Animation|Comedy|Family|Fantasy,7.9,John Lasseter,Tom Hanks,John Ratzenberger,Wayne Knight,863.0,IMDb + TMDB
2,40,Back to the Future (1985),0.548,1985,Adventure|Comedy|Sci-Fi,8.5,Robert Zemeckis,Lea Thompson,Thomas F. Wilson,J.J. Cohen,105.0,IMDb + TMDB
3,13,Forrest Gump (1994),0.544,1994,Comedy|Drama,8.8,Robert Zemeckis,Tom Hanks,Siobhan Fallon Hogan,Sam Anderson,13.0,IMDb + TMDB
4,17,Jurassic Park (1993),0.540,1993,Adventure|Sci-Fi|Thriller,8.1,Steven Spielberg,Wayne Knight,Ariana Richards,Bob Peck,329.0,IMDb + TMDB
5,37,Star Wars: Episode VI - Return of the Jedi (1983),0.539,1983,Action|Adventure|Fantasy|Sci-Fi,8.4,Richard Marquand,Harrison Ford,Ian McDiarmid,Kenny Baker,NaN,IMDb only
6,192,Independence Day (a.k.a. ID4) (1996),0.538,1996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No metadata
7,14,"Lion King, The (1994)",0.531,1994,Adventure|Animation|Drama|Family|Musical,8.5,Roger Allers,Matthew Broderick,Nathan Lane,Niketa Calame,8587.0,IMDb + TMDB
8,23,Aladdin (1992),0.529,1992,Adventure|Animation|Comedy|Family|Fantasy|Musi...,8.0,Ron Clements,Robin Williams,Frank Welker,Scott Weinger,812.0,IMDb + TMDB
9,33,Star Wars: Episode V - The Empire Strikes Back...,0.513,1980,Action|Adventure|Fantasy|Sci-Fi,8.8,Irvin Kershner,Harrison Ford,Kenny Baker,Anthony Daniels,NaN,IMDb only


In [64]:
def get_recommendations(movie_name, n_recommendations=10):

    recommendations = recommend_with_metadata(
        movie_name,
        n_recommendations
    )

    if recommendations is None:
        return None

    return clean_recommendations(recommendations)

In [65]:
get_recommendations("Toy Story (1995)")

,canonical_movie_id,Movie_Name,Similarity,year,genres,imdb_rating,director,actor_1,actor_2,actor_3,tmdb_id,metadata_status
0,7,Star Wars: Episode IV - A New Hope (1977),0.567,1977,Action|Adventure|Fantasy|Sci-Fi,8.7,George Lucas,Harrison Ford,Peter Cushing,Kenny Baker,NaN,IMDb only
1,105,Toy Story 2 (1999),0.564,1999,Adventure|Animation|Comedy|Family|Fantasy,7.9,John Lasseter,Tom Hanks,John Ratzenberger,Wayne Knight,863.0,IMDb + TMDB
2,40,Back to the Future (1985),0.548,1985,Adventure|Comedy|Sci-Fi,8.5,Robert Zemeckis,Lea Thompson,Thomas F. Wilson,J.J. Cohen,105.0,IMDb + TMDB
3,13,Forrest Gump (1994),0.544,1994,Comedy|Drama,8.8,Robert Zemeckis,Tom Hanks,Siobhan Fallon Hogan,Sam Anderson,13.0,IMDb + TMDB
4,17,Jurassic Park (1993),0.540,1993,Adventure|Sci-Fi|Thriller,8.1,Steven Spielberg,Wayne Knight,Ariana Richards,Bob Peck,329.0,IMDb + TMDB
5,37,Star Wars: Episode VI - Return of the Jedi (1983),0.539,1983,Action|Adventure|Fantasy|Sci-Fi,8.4,Richard Marquand,Harrison Ford,Ian McDiarmid,Kenny Baker,NaN,IMDb only
6,192,Independence Day (a.k.a. ID4) (1996),0.538,1996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No metadata
7,14,"Lion King, The (1994)",0.531,1994,Adventure|Animation|Drama|Family|Musical,8.5,Roger Allers,Matthew Broderick,Nathan Lane,Niketa Calame,8587.0,IMDb + TMDB
8,23,Aladdin (1992),0.529,1992,Adventure|Animation|Comedy|Family|Fantasy|Musi...,8.0,Ron Clements,Robin Williams,Frank Welker,Scott Weinger,812.0,IMDb + TMDB
9,33,Star Wars: Episode V - The Empire Strikes Back...,0.513,1980,Action|Adventure|Fantasy|Sci-Fi,8.8,Irvin Kershner,Harrison Ford,Kenny Baker,Anthony Daniels,NaN,IMDb only


In [66]:
def prepare_recommendations(movie_name, n_recommendations=10):

    result = get_recommendations(
        movie_name,
        n_recommendations
    )

    if result is None:
        return None

    result = result.copy()

    # Add recommendation rank
    result.insert(
        0,
        "Rank",
        range(1, len(result) + 1)
    )

    return result

In [67]:
final_recommendations = prepare_recommendations(
    "Toy Story (1995)"
)

final_recommendations

,Rank,canonical_movie_id,Movie_Name,Similarity,year,genres,imdb_rating,director,actor_1,actor_2,actor_3,tmdb_id,metadata_status
0,1,7,Star Wars: Episode IV - A New Hope (1977),0.567,1977,Action|Adventure|Fantasy|Sci-Fi,8.7,George Lucas,Harrison Ford,Peter Cushing,Kenny Baker,NaN,IMDb only
1,2,105,Toy Story 2 (1999),0.564,1999,Adventure|Animation|Comedy|Family|Fantasy,7.9,John Lasseter,Tom Hanks,John Ratzenberger,Wayne Knight,863.0,IMDb + TMDB
2,3,40,Back to the Future (1985),0.548,1985,Adventure|Comedy|Sci-Fi,8.5,Robert Zemeckis,Lea Thompson,Thomas F. Wilson,J.J. Cohen,105.0,IMDb + TMDB
3,4,13,Forrest Gump (1994),0.544,1994,Comedy|Drama,8.8,Robert Zemeckis,Tom Hanks,Siobhan Fallon Hogan,Sam Anderson,13.0,IMDb + TMDB
4,5,17,Jurassic Park (1993),0.540,1993,Adventure|Sci-Fi|Thriller,8.1,Steven Spielberg,Wayne Knight,Ariana Richards,Bob Peck,329.0,IMDb + TMDB
5,6,37,Star Wars: Episode VI - Return of the Jedi (1983),0.539,1983,Action|Adventure|Fantasy|Sci-Fi,8.4,Richard Marquand,Harrison Ford,Ian McDiarmid,Kenny Baker,NaN,IMDb only
6,7,192,Independence Day (a.k.a. ID4) (1996),0.538,1996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No metadata
7,8,14,"Lion King, The (1994)",0.531,1994,Adventure|Animation|Drama|Family|Musical,8.5,Roger Allers,Matthew Broderick,Nathan Lane,Niketa Calame,8587.0,IMDb + TMDB
8,9,23,Aladdin (1992),0.529,1992,Adventure|Animation|Comedy|Family|Fantasy|Musi...,8.0,Ron Clements,Robin Williams,Frank Welker,Scott Weinger,812.0,IMDb + TMDB
9,10,33,Star Wars: Episode V - The Empire Strikes Back...,0.513,1980,Action|Adventure|Fantasy|Sci-Fi,8.8,Irvin Kershner,Harrison Ford,Kenny Baker,Anthony Daniels,NaN,IMDb only


In [68]:
def application_recommendations(movie_name, n_recommendations=10):

    result = get_recommendations(
        movie_name,
        n_recommendations
    )

    if result is None:
        return None

    result = result.copy()

    result.insert(
        0,
        "Rank",
        range(1, len(result) + 1)
    )

    columns = [
        "Rank",
        "canonical_movie_id",
        "Movie_Name",
        "Similarity",
        "year",
        "genres",
        "imdb_rating",
        "director",
        "actor_1",
        "actor_2",
        "actor_3",
        "tmdb_id",
        "metadata_status"
    ]

    columns = [
        column
        for column in columns
        if column in result.columns
    ]

    return result[columns]

In [69]:
application_result = application_recommendations(
    "Toy Story (1995)"
)

application_result

,Rank,canonical_movie_id,Movie_Name,Similarity,year,genres,imdb_rating,director,actor_1,actor_2,actor_3,tmdb_id,metadata_status
0,1,7,Star Wars: Episode IV - A New Hope (1977),0.567,1977,Action|Adventure|Fantasy|Sci-Fi,8.7,George Lucas,Harrison Ford,Peter Cushing,Kenny Baker,NaN,IMDb only
1,2,105,Toy Story 2 (1999),0.564,1999,Adventure|Animation|Comedy|Family|Fantasy,7.9,John Lasseter,Tom Hanks,John Ratzenberger,Wayne Knight,863.0,IMDb + TMDB
2,3,40,Back to the Future (1985),0.548,1985,Adventure|Comedy|Sci-Fi,8.5,Robert Zemeckis,Lea Thompson,Thomas F. Wilson,J.J. Cohen,105.0,IMDb + TMDB
3,4,13,Forrest Gump (1994),0.544,1994,Comedy|Drama,8.8,Robert Zemeckis,Tom Hanks,Siobhan Fallon Hogan,Sam Anderson,13.0,IMDb + TMDB
4,5,17,Jurassic Park (1993),0.540,1993,Adventure|Sci-Fi|Thriller,8.1,Steven Spielberg,Wayne Knight,Ariana Richards,Bob Peck,329.0,IMDb + TMDB
5,6,37,Star Wars: Episode VI - Return of the Jedi (1983),0.539,1983,Action|Adventure|Fantasy|Sci-Fi,8.4,Richard Marquand,Harrison Ford,Ian McDiarmid,Kenny Baker,NaN,IMDb only
6,7,192,Independence Day (a.k.a. ID4) (1996),0.538,1996,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No metadata
7,8,14,"Lion King, The (1994)",0.531,1994,Adventure|Animation|Drama|Family|Musical,8.5,Roger Allers,Matthew Broderick,Nathan Lane,Niketa Calame,8587.0,IMDb + TMDB
8,9,23,Aladdin (1992),0.529,1992,Adventure|Animation|Comedy|Family|Fantasy|Musi...,8.0,Ron Clements,Robin Williams,Frank Welker,Scott Weinger,812.0,IMDb + TMDB
9,10,33,Star Wars: Episode V - The Empire Strikes Back...,0.513,1980,Action|Adventure|Fantasy|Sci-Fi,8.8,Irvin Kershner,Harrison Ford,Kenny Baker,Anthony Daniels,NaN,IMDb only


In [70]:
NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11
)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=11)

In [71]:
movie_rating_counts = (
    ratings_ml.groupby("Movie_Name")
    .size()
    .sort_values(ascending=False)
)

print("Movies:", len(movie_rating_counts))
print("\nMost rated movies:")
print(movie_rating_counts.head(10))

print("\nLeast rated movies:")
print(movie_rating_counts.tail(10))

Movies: 3794

Most rated movies:
Movie_Name
Forrest Gump (1994)                          81491
Shawshank Redemption, The (1994)             81482
Pulp Fiction (1994)                          79672
Silence of the Lambs, The (1991)             74127
Matrix, The (1999)                           72674
Star Wars: Episode IV - A New Hope (1977)    68717
Jurassic Park (1993)                         64144
Schindler's List (1993)                      60411
Braveheart (1995)                            59184
Fight Club (1999)                            58773
dtype: int64

Least rated movies:
Movie_Name
Murder by Death (1976)              1003
Jack Ryan: Shadow Recruit (2014)    1003
Interstate 60 (2002)                1002
Last Waltz, The (1978)              1002
Pet Sematary II (1992)              1001
Child's Play 3 (1991)               1001
Darkest Hour (2017)                 1000
Black Mass (2015)                   1000
Farinelli: il castrato (1994)       1000
Stakeout (1987)                 

In [72]:
print("\nRating count statistics:")
print(movie_rating_counts.describe())


Rating count statistics:
count     3794.00000
mean      5836.21165
std       7999.34351
min       1000.00000
25%       1586.25000
50%       2855.50000
75%       6579.50000
max      81491.00000
dtype: float64


In [73]:
test_movies = [
    "Toy Story (1995)",
    "Star Wars: Episode IV - A New Hope (1977)",
    "Forrest Gump (1994)",
    "Jurassic Park (1993)",
    "The Godfather (1972)"
]

In [74]:
test_result = get_recommendations(
    "Toy Story (1995)",
    n_recommendations=10
)

popularity_test = test_result[
    ["Movie_Name", "Similarity"]
].merge(
    movie_rating_counts.rename("rating_count"),
    left_on="Movie_Name",
    right_index=True,
    how="left"
)

popularity_test

,Movie_Name,Similarity,rating_count
0,Star Wars: Episode IV - A New Hope (1977),0.567,68717
1,Toy Story 2 (1999),0.564,26536
2,Back to the Future (1985),0.548,49595
3,Forrest Gump (1994),0.544,81491
4,Jurassic Park (1993),0.540,64144
5,Star Wars: Episode VI - Return of the Jedi (1983),0.539,54917
6,Independence Day (a.k.a. ID4) (1996),0.538,48902
7,"Lion King, The (1994)",0.531,42745
8,Aladdin (1992),0.529,43387
9,Star Wars: Episode V - The Empire Strikes Back...,0.513,57361


In [75]:
print(
    "Average ratings of recommended movies:",
    round(popularity_test["rating_count"].mean(), 0)
)

print(
    "Average ratings across all model movies:",
    round(movie_rating_counts.mean(), 0)
)

Average ratings of recommended movies: 53780.0
Average ratings across all model movies: 5836.0


In [76]:
print(
    "Median ratings of recommended movies:",
    popularity_test["rating_count"].median()
)

print(
    "Median ratings across all model movies:",
    movie_rating_counts.median()
)

print(
    "Recommendation popularity ratio:",
    round(
        popularity_test["rating_count"].median()
        / movie_rating_counts.median(),
        2
    )
)

Median ratings of recommended movies: 52256.0
Median ratings across all model movies: 2855.5
Recommendation popularity ratio: 18.3


In [77]:
import numpy as np

def popularity_adjusted_score(similarity, rating_count, alpha=0.10):
    popularity_factor = np.log1p(rating_count)

    return similarity / (
        1 + alpha * popularity_factor
    )

In [78]:
popularity_test = popularity_test.copy()

popularity_test["adjusted_score"] = popularity_test.apply(
    lambda row: popularity_adjusted_score(
        row["Similarity"],
        row["rating_count"]
    ),
    axis=1
)

popularity_test = popularity_test.sort_values(
    "adjusted_score",
    ascending=False
).reset_index(drop=True)

popularity_test[
    [
        "Movie_Name",
        "Similarity",
        "rating_count",
        "adjusted_score"
    ]
]

,Movie_Name,Similarity,rating_count,adjusted_score
0,Toy Story 2 (1999),0.564,26536,0.279397
1,Star Wars: Episode IV - A New Hope (1977),0.567,68717,0.268240
2,Back to the Future (1985),0.548,49595,0.263314
3,Independence Day (a.k.a. ID4) (1996),0.538,48902,0.258684
4,Star Wars: Episode VI - Return of the Jedi (1983),0.539,54917,0.257727
5,"Lion King, The (1994)",0.531,42745,0.256981
6,Jurassic Park (1993),0.540,64144,0.256302
7,Aladdin (1992),0.529,43387,0.255828
8,Forrest Gump (1994),0.544,81491,0.255300
9,Star Wars: Episode V - The Empire Strikes Back...,0.513,57361,0.244785


In [79]:
adjusted_counts = popularity_test["rating_count"]

print(
    "Adjusted average popularity:",
    round(adjusted_counts.mean())
)

print(
    "Adjusted median popularity:",
    round(adjusted_counts.median())
)

Adjusted average popularity: 53780
Adjusted median popularity: 52256


In [80]:
def get_candidates(movie_name, n_candidates=100):

    if movie_name not in movie_encoder.classes_:
        print("Movie not found in the recommendation model.")
        return None

    movie_index = movie_encoder.transform([movie_name])[0]

    distances, indices = model.kneighbors(
        sparse_matrix.T[movie_index],
        n_neighbors=n_candidates + 1
    )

    candidates = []

    for distance, index in zip(
        distances[0][1:],
        indices[0][1:]
    ):
        recommended_movie = movie_encoder.inverse_transform(
            [index]
        )[0]

        similarity = 1 - distance

        candidates.append({
            "Movie_Name": recommended_movie,
            "Similarity": similarity
        })

    candidates_df = pd.DataFrame(candidates)

    candidates_df = candidates_df.merge(
        movie_rating_counts.rename("rating_count"),
        left_on="Movie_Name",
        right_index=True,
        how="left"
    )

    return candidates_df

In [81]:
candidates = get_candidates(
    "Toy Story (1995)",
    n_candidates=100
)

print("Candidates:", len(candidates))
candidates.head(10)

Candidates: 100


,Movie_Name,Similarity,rating_count
0,Star Wars: Episode IV - A New Hope (1977),0.567278,68717
1,Toy Story 2 (1999),0.563549,26536
2,Back to the Future (1985),0.548156,49595
3,Forrest Gump (1994),0.543782,81491
4,Jurassic Park (1993),0.539750,64144
5,Star Wars: Episode VI - Return of the Jedi (1983),0.538781,54917
6,Independence Day (a.k.a. ID4) (1996),0.537721,48902
7,"Lion King, The (1994)",0.530698,42745
8,Aladdin (1992),0.528697,43387
9,Star Wars: Episode V - The Empire Strikes Back...,0.513224,57361


In [82]:
candidates["adjusted_score"] = candidates.apply(
    lambda row: popularity_adjusted_score(
        row["Similarity"],
        row["rating_count"]
    ),
    axis=1
)

In [83]:
adjusted_recommendations = (
    candidates
    .sort_values(
        "adjusted_score",
        ascending=False
    )
    .head(10)
    .reset_index(drop=True)
)

adjusted_recommendations

,Movie_Name,Similarity,rating_count,adjusted_score
0,Toy Story 2 (1999),0.563549,26536,0.279174
1,Star Wars: Episode IV - A New Hope (1977),0.567278,68717,0.268372
2,Back to the Future (1985),0.548156,49595,0.263389
3,Independence Day (a.k.a. ID4) (1996),0.537721,48902,0.258550
4,Star Wars: Episode VI - Return of the Jedi (1983),0.538781,54917,0.257622
5,"Lion King, The (1994)",0.530698,42745,0.256835
6,Jurassic Park (1993),0.539750,64144,0.256183
7,Aladdin (1992),0.528697,43387,0.255682
8,Forrest Gump (1994),0.543782,81491,0.255198
9,Shrek (2001),0.510188,42303,0.247033


In [84]:
print(
    "Original median popularity:",
    popularity_test["rating_count"].median()
)

print(
    "Adjusted median popularity:",
    adjusted_recommendations["rating_count"].median()
)

print(
    "Adjusted recommendations:"
)

print(
    adjusted_recommendations[
        [
            "Movie_Name",
            "Similarity",
            "rating_count",
            "adjusted_score"
        ]
    ]
)

Original median popularity: 52256.0
Adjusted median popularity: 49248.5
Adjusted recommendations:
                                          Movie_Name  Similarity  \
0                                 Toy Story 2 (1999)    0.563549   
1          Star Wars: Episode IV - A New Hope (1977)    0.567278   
2                          Back to the Future (1985)    0.548156   
3               Independence Day (a.k.a. ID4) (1996)    0.537721   
4  Star Wars: Episode VI - Return of the Jedi (1983)    0.538781   
5                              Lion King, The (1994)    0.530698   
6                               Jurassic Park (1993)    0.539750   
7                                     Aladdin (1992)    0.528697   
8                                Forrest Gump (1994)    0.543782   
9                                       Shrek (2001)    0.510188   

   rating_count  adjusted_score  
0         26536        0.279174  
1         68717        0.268372  
2         49595        0.263389  
3         48902  

In [85]:
alphas = [0, 0.05, 0.10, 0.20]

test_movies = [
    "Toy Story (1995)",
    "Star Wars: Episode IV - A New Hope (1977)",
    "Forrest Gump (1994)",
    "Jurassic Park (1993)",
    "Pulp Fiction (1994)"
]

results = []

for movie in test_movies:

    candidates = get_candidates(
        movie,
        n_candidates=100
    )

    for alpha in alphas:

        temp = candidates.copy()

        temp["adjusted_score"] = temp.apply(
            lambda row: popularity_adjusted_score(
                row["Similarity"],
                row["rating_count"],
                alpha=alpha
            ),
            axis=1
        )

        top10 = (
            temp
            .sort_values("adjusted_score", ascending=False)
            .head(10)
        )

        results.append({
            "movie": movie,
            "alpha": alpha,
            "median_popularity": top10["rating_count"].median(),
            "average_similarity": top10["Similarity"].mean()
        })

evaluation_df = pd.DataFrame(results)

evaluation_df

,movie,alpha,median_popularity,average_similarity
0,Toy Story (1995),0.00,52256.0,0.541164
1,Toy Story (1995),0.05,49248.5,0.540860
2,Toy Story (1995),0.10,49248.5,0.540860
3,Toy Story (1995),0.20,49248.5,0.540293
4,Star Wars: Episode IV - A New Hope (1977),0.00,54796.0,0.645146
5,Star Wars: Episode IV - A New Hope (1977),0.05,54796.0,0.645146
6,Star Wars: Episode IV - A New Hope (1977),0.10,52135.0,0.643954
7,Star Wars: Episode IV - A New Hope (1977),0.20,52135.0,0.643954
8,Forrest Gump (1994),0.00,62277.5,0.618841
9,Forrest Gump (1994),0.05,59797.5,0.617991


In [86]:
evaluation_summary = (
    evaluation_df
    .groupby("alpha")
    .agg({
        "median_popularity": "mean",
        "average_similarity": "mean"
    })
    .reset_index()
)

evaluation_summary

,alpha,median_popularity,average_similarity
0,0.00,55282.6,0.612649
1,0.05,54185.1,0.612342
2,0.10,53652.9,0.612103
3,0.20,53579.3,0.611711


In [87]:
large_candidates = get_candidates(
    "Toy Story (1995)",
    n_candidates=500
)

print("Candidates:", len(large_candidates))

Candidates: 500


In [88]:
large_candidates["adjusted_score"] = large_candidates.apply(
    lambda row: popularity_adjusted_score(
        row["Similarity"],
        row["rating_count"],
        alpha=0.10
    ),
    axis=1
)

large_adjusted = (
    large_candidates
    .sort_values("adjusted_score", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

large_adjusted[
    [
        "Movie_Name",
        "Similarity",
        "rating_count",
        "adjusted_score"
    ]
]

,Movie_Name,Similarity,rating_count,adjusted_score
0,Toy Story 2 (1999),0.563549,26536,0.279174
1,Star Wars: Episode IV - A New Hope (1977),0.567278,68717,0.268372
2,Back to the Future (1985),0.548156,49595,0.263389
3,Independence Day (a.k.a. ID4) (1996),0.537721,48902,0.258550
4,Star Wars: Episode VI - Return of the Jedi (1983),0.538781,54917,0.257622
5,"Lion King, The (1994)",0.530698,42745,0.256835
6,Jurassic Park (1993),0.539750,64144,0.256183
7,Aladdin (1992),0.528697,43387,0.255682
8,Forrest Gump (1994),0.543782,81491,0.255198
9,Shrek (2001),0.510188,42303,0.247033


In [89]:
print(
    "500-candidate adjusted median popularity:",
    large_adjusted["rating_count"].median()
)

print(
    "500-candidate adjusted average similarity:",
    large_adjusted["Similarity"].mean()
)

500-candidate adjusted median popularity: 49248.5
500-candidate adjusted average similarity: 0.5408601


In [90]:
from pathlib import Path
import joblib
from scipy.sparse import save_npz

# Create model artifact folder
artifact_dir = Path("model") / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)

# 1. Save trained KNN model
joblib.dump(
    model,
    artifact_dir / "movie_model.pkl"
)

# 2. Save movie encoder
joblib.dump(
    movie_encoder,
    artifact_dir / "movie_encoder.pkl"
)

# 3. Save the sparse user-movie matrix
save_npz(
    artifact_dir / "rating_matrix.npz",
    sparse_matrix
)

# 4. Save movie master
movie_master.to_csv(
    artifact_dir / "movie_master.csv",
    index=False
)

print("Model artifacts saved successfully!")
print("Location:", artifact_dir.resolve())

Model artifacts saved successfully!
Location: C:\Users\karnik YOGA\OneDrive\Desktop\ml projects\notebooks\model\artifacts


In [91]:
from pathlib import Path

artifact_dir = Path("../model/artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

print(artifact_dir.resolve())

C:\Users\karnik YOGA\OneDrive\Desktop\ml projects\model\artifacts


In [92]:
import joblib
from scipy.sparse import save_npz

joblib.dump(
    model,
    artifact_dir / "movie_model.pkl"
)

joblib.dump(
    movie_encoder,
    artifact_dir / "movie_encoder.pkl"
)

save_npz(
    artifact_dir / "rating_matrix.npz",
    sparse_matrix
)

movie_master.to_csv(
    artifact_dir / "movie_master.csv",
    index=False
)

print("Artifacts saved to the actual model folder!")

Artifacts saved to the actual model folder!
